In [20]:
import pandas as pd
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_postgres import PGVector
from IPython.display import Markdown, display

In [21]:
load_dotenv(".env")

LAUKI_DATABASE_URL = os.getenv("LAUKI_DATABASE_URL")

## Step 1. Load the CSV

In [ ]:
df = pd.read_csv("lauki_qna.csv")

documents = []

for _, row in df.iterrows():

    documents.append(
        Document(
            page_content=f"""
            Question:
            {row['question']}

            Answer:
            {row['answer']}
        """,
            metadata={
                "source": "lauki_faq.csv",
                "question": row["question"],
            },
        )
    )

print(f"Loaded {len(documents)} documents.")

Loaded 75 documents.


## Step 2. Create Embeddings

In [23]:
#########################################
# 2. Create embeddings
#########################################

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Step 3. Store in pgvector

In [24]:
#################################################
# 4. PGVector
#################################################

# Add the document chunks to the "vector store" using OpenAIEmbeddings
# vectorstore = InMemoryVectorStore.from_documents(
#     documents=chunks,
#     embedding=OpenAIEmbeddings(),
# )

vectorstore = PGVector(
    embeddings=embeddings,
    collection_name="lauki_faq",
    connection=LAUKI_DATABASE_URL,
    use_jsonb=True,
)

In [25]:
#################################################
# 5. Insert documents
#################################################

vectorstore.add_documents(documents)

print("Documents stored!")

Documents stored!


## Step 4. Ask Questions

In [26]:
query = "Can I convert my SIM into an eSIM?"

results = vectorstore.similarity_search_with_score(
    query=query,
    k=3,
)

for doc, score in results:

    print("=" * 60)
    print(f"Cosine Distance: {score:.4f}")
    print(doc.page_content)

Cosine Distance: 0.3597

            Question:
            How do I switch from physical SIM to eSIM?

            Answer:
            Authenticate in the portal, request conversion, validate identity, and scan the generated QR on the device. The physical SIM deactivates immediately after profile installation.
        
Cosine Distance: 0.3602

            Question:
            How do I switch from physical SIM to eSIM?

            Answer:
            Authenticate in the portal, request conversion, validate identity, and scan the generated QR on the device. The physical SIM deactivates immediately after profile installation.
        
Cosine Distance: 0.4799

            Question:
            How do I move my service to a new device?

            Answer:
            To move service, power off the old device, insert the physical SIM or scan the eSIM QR code on the new one, then restart. The network detects the change via IMSI and automatically reprofiles capabilities based on the new dev

## Step 5. Convert Distance to Similarity

In [27]:
for doc, distance in results:

    similarity = 1 - distance

    print(f"Similarity: {similarity:.2%}")
    print(doc.metadata["question"])

Similarity: 64.03%
How do I switch from physical SIM to eSIM?
Similarity: 63.98%
How do I switch from physical SIM to eSIM?
Similarity: 52.01%
How do I move my service to a new device?


## Step 6. Generate an Answer

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

context = "\n\n".join(
    doc.page_content for doc, _ in results
)

prompt = f"""
You are a customer support assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print("\nAnswer:\n")
display(Markdown(response.content))


Answer



Authenticate in the portal, request conversion, validate identity, and scan the generated QR on the device. The physical SIM deactivates immediately after profile installation.

In [31]:
query = "What is the process for porting my number to a different network?"

results = vectorstore.similarity_search_with_score(
    query=query,
    k=3,
)

llm = ChatOpenAI(model="gpt-4o-mini")

context = "\n\n".join(
    doc.page_content for doc, _ in results
)

prompt = f"""
You are a customer support assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print("\nAnswer\n")
display(Markdown(response.content))


Answer



Submit a porting request using the port code issued by the current carrier. Lauki Phones validates identity, initiates number transfer, and schedules activation.

In [32]:
query = "Does Lauki Phones support WiFi Calling?"

results = vectorstore.similarity_search_with_score(
    query=query,
    k=3,
)

llm = ChatOpenAI(model="gpt-4o-mini")

context = "\n\n".join(
    doc.page_content for doc, _ in results
)

prompt = f"""
You are a customer support assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print("\nAnswer\n")
display(Markdown(response.content))


Answer



Supported for most current smartphones. Calls route through secure IMS tunnels when cellular coverage is weak, maintaining voice quality and call continuity.

In [33]:
query = "Are there late payment fees?"

results = vectorstore.similarity_search_with_score(
    query=query,
    k=3,
)

llm = ChatOpenAI(model="gpt-4o-mini")

context = "\n\n".join(
    doc.page_content for doc, _ in results
)

prompt = f"""
You are a customer support assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print("\nAnswer\n")
display(Markdown(response.content))


Answer



Yes, late payment fees apply after the grace period: a fixed fee of ₹50-100 (region-dependent) for the first overdue instance, escalating to 1.5% daily interest on the outstanding amount thereafter, in line with RBI guidelines and region-specific billing regulations like TRAI mandates in India. Fees are waived for first-time delays under 5 days or proven technical glitches, with full transparency in the invoice summary.